# Itertools => Filtering & Slicing Iterators

These tools select part of a stream without building a list.

| Function | Purpose | Example |
|---|---|---|
| `islice(it, start, stop, step)` | Slice any iterable | `islice("ABCDEFG", 2, 4)` gives C, D |
| `takewhile(pred, it)` | Take items while the test passes | `takewhile(lambda n: n < 3, [1, 2, 5, 1])` gives 1, 2 |
| `dropwhile(pred, it)` | Skip items while the test passes | `dropwhile(lambda n: n < 3, [1, 2, 5, 1])` gives 5, 1 |
| `filterfalse(pred, it)` | Keep items where the test fails | `filterfalse(lambda n: n < 3, [1, 5])` gives 5 |
| `compress(data, selectors)` | Keep items using a mask | `compress("ABC", [1, 0, 1])` gives A, C |

---

## `islice(iterable, stop)` and `islice(iterable, start, stop[, step])`

Slices **any iterable**, including generators and infinite iterators.

```python
from itertools import islice

list(islice("ABCDEFG", 2))            # ['A', 'B']
list(islice("ABCDEFG", 2, 4))         # ['C', 'D']
list(islice("ABCDEFG", 2, None))      # ['C', 'D', 'E', 'F', 'G']
list(islice("ABCDEFG", 0, None, 2))   # ['A', 'C', 'E', 'G']
```

- **No negative** `start`, `stop` or `step`.
- It works by consuming items, so it **advances** the source iterator.
- Slicing an iterator twice continues where the first slice stopped.

## `takewhile(predicate, iterable)`

Yields items **while** the predicate is true, then stops for good.

```python
list(takewhile(lambda n: n < 3, [1, 2, 5, 1, 2]))   # [1, 2]
```

The first item that fails the test is **consumed and lost** from the source iterator.

## `dropwhile(predicate, iterable)`

Skips items **while** the predicate is true, then yields **everything** that follows, even items that would pass the test again.

```python
list(dropwhile(lambda n: n < 3, [1, 2, 5, 1, 2]))   # [5, 1, 2]
```

It produces no output until the predicate first fails, so its start-up may be slow.

## `filterfalse(predicate, iterable)`

The opposite of `filter()`: keeps items where the predicate is **false**.

```python
list(filterfalse(lambda n: n < 3, [1, 2, 5, 1, 2]))   # [5]
```

With `None` as the predicate, it keeps the items that are falsy (`0`, `""`, `None`, ...).

## `compress(data, selectors)`

Keeps `data` items whose matching selector is truthy. It stops at the shorter input.

```python
list(compress("ABCDEF", [1, 0, 1, 0, 1, 1]))   # ['A', 'C', 'E', 'F']
```

## Which One Do I Need?

| Need | Tool |
|---|---|
| First `n` items, or a range of items | `islice` |
| Keep the leading items that pass a test | `takewhile` |
| Skip the leading items that pass a test | `dropwhile` |
| Keep items that fail a test | `filterfalse` |
| Keep items using a mask | `compress` |
| Keep every matching item anywhere | `filter()` or a comprehension |

## Handy Patterns

```python
def take(n, iterable):
    return list(islice(iterable, n))

def nth(iterable, n, default=None):
    return next(islice(iterable, n, None), default)
```

## Key Rules

- `takewhile` and `dropwhile` only look at the **start** of the stream. `filter()` checks every item.
- `takewhile` consumes the first failing item.
- `islice` never uses negative numbers.

## Source

https://docs.python.org/3/library/itertools.html#itertools.islice

https://docs.python.org/3/library/itertools.html#itertools.takewhile

In [ ]:
from itertools import islice, takewhile, dropwhile, filterfalse, compress, count

# islice: slicing any iterable
print(list(islice("ABCDEFG", 2)))              # ['A', 'B']
print(list(islice("ABCDEFG", 2, 4)))           # ['C', 'D']
print(list(islice("ABCDEFG", 2, None)))        # ['C', 'D', 'E', 'F', 'G']
print(list(islice("ABCDEFG", 0, None, 2)))     # ['A', 'C', 'E', 'G']

# islice advances the source iterator
source = iter(range(10))
print(list(islice(source, 3)))                 # [0, 1, 2]
print(list(islice(source, 3)))                 # [3, 4, 5]: continues where it stopped

# islice works on infinite iterators
print(list(islice(count(100), 3)))             # [100, 101, 102]

# takewhile: stop for good at the first failure
print(list(takewhile(lambda n: n < 3, [1, 2, 5, 1, 2])))    # [1, 2]

# ... and the failing item is consumed
stream = iter([1, 2, 5, 1, 2])
print(list(takewhile(lambda n: n < 3, stream)))             # [1, 2]
print(list(stream))                                          # [1, 2]: the 5 is gone

# dropwhile: skip the start, then yield everything
print(list(dropwhile(lambda n: n < 3, [1, 2, 5, 1, 2])))    # [5, 1, 2]

# filterfalse: the opposite of filter
print(list(filterfalse(lambda n: n < 3, [1, 2, 5, 1, 2])))  # [5]
print(list(filter(lambda n: n < 3, [1, 2, 5, 1, 2])))       # [1, 2, 1, 2]
print(list(filterfalse(None, [0, 1, "", "a", None])))       # [0, '', None]

# compress: keep items using a mask
print(list(compress("ABCDEF", [1, 0, 1, 0, 1, 1])))         # ['A', 'C', 'E', 'F']
data = [5, 1, 8, 3]
print(list(compress(data, (n > 2 for n in data))))          # [5, 8, 3]

# Handy patterns
def take(n, iterable):
    return list(islice(iterable, n))

def nth(iterable, n, default=None):
    return next(islice(iterable, n, None), default)

print(take(3, count(1)))                         # [1, 2, 3]
print(nth("ABCDEF", 2), nth("ABC", 10, "none"))  # C none